# Milestone 1: Exploratory Data Analysis & Quality Inspection
## AI-Based Transformer Load, Overload and Efficiency Prediction System

### Objectives:
1. **Dataset Overview**: Rows, columns, types, date ranges, and sampling continuity.
2. **Data Quality Assessment**: Missing value distribution, anomalies, constant-value signals, and time gaps.
3. **Transformer Parameter Analysis**: T1, T2, and T3 load distribution, voltage, current, and temperature dynamics.
4. **Festival & Uplift Impact**: Empirical comparison between Normal Day and Festival operational regimes.
5. **ML Readiness Evaluation**: Identification of target variables, candidate features, lag creation constraints, and physics boundaries.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

os.environ['MPLCONFIGDIR'] = os.path.abspath('../.matplotlib_cache')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

DATA_PATH = '../data/raw/transformer_hourly_festival_with_temperature.csv'
df = pd.read_csv(DATA_PATH)
print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### 1. Dataset Dimensions and Schema

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime_IST'])
date_min = df['datetime'].min()
date_max = df['datetime'].max()
print(f"Temporal span: {date_min} to {date_max}")
print(f"Total days: {(date_max - date_min).days + 1}")
print(f"Sampling diffs count:\n{df['datetime'].diff().value_counts()}")
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate timestamps: {df.duplicated(subset=['datetime_IST']).sum()}")

### 2. Missing Value & Outage Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing, 'Missing_Pct': missing_pct})
missing_df[missing_df['Missing_Count'] > 0]

### 3. Constant and Invariant Parameters Detection

In [ ]:
for t in ['t1', 't2', 't3']:
    print(f"{t.upper()} Constant Check:")
    print(f"  Voltage: {df[f'{t}_voltage_V'].dropna().unique()}")
    print(f"  Power Factor: {df[f'{t}_pf'].dropna().unique()}")
    print(f"  Frequency: {df[f'{t}_frequency_Hz'].dropna().unique()}")

### 4. Transformer Load & Operating Dynamics

In [ ]:
summary = []
for t in ['t1', 't2', 't3']:
    p = df[f'{t}_power_W'].dropna()
    l = df[f'{t}_load_pct'].dropna()
    temp = df[f'{t}_oil_temp_C'].dropna()
    summary.append({
        'Transformer': t.upper(),
        'Mean Active Power (kW)': p.mean() / 1000,
        'Max Active Power (kW)': p.max() / 1000,
        'Mean Load (%)': l.mean(),
        'Max Load (%)': l.max(),
        'Overload Hours (>100%)': (l > 100.0).sum(),
        'Severe Overload (>120%)': (l > 120.0).sum(),
        'Mean Oil Temp (°C)': temp.mean(),
        'Max Oil Temp (°C)': temp.max()
    })
pd.DataFrame(summary).set_index('Transformer')

### 5. Festival vs Normal Day Analysis

In [ ]:
is_fest = df['festival'] != 'Normal day'
print(f"Festival hours: {is_fest.sum()} ({is_fest.mean()*100:.2f}%)")
print(f"Festival distribution:\n{df['festival'].value_counts()}\n")

fest_comp = []
for t in ['t1', 't2', 't3']:
    norm_p = df.loc[~is_fest, f'{t}_power_W'].mean() / 1000
    fest_p = df.loc[is_fest, f'{t}_power_W'].mean() / 1000
    fest_comp.append({
        'Transformer': t.upper(),
        'Normal Mean (kW)': norm_p,
        'Festival Mean (kW)': fest_p,
        'Empirical Uplift (%)': ((fest_p - norm_p) / norm_p) * 100
    })
pd.DataFrame(fest_comp).set_index('Transformer')

### 6. Visualizations Overview

In [ ]:
# Display generated charts from visualizations directory
from IPython.display import Image, display
vis_files = [
    '../visualizations/01_hourly_load_trends.png',
    '../visualizations/02_daily_monthly_load_trends.png',
    '../visualizations/03_festival_vs_non_festival.png',
    '../visualizations/04_transformer_wise_comparison.png',
    '../visualizations/05_missing_values_timeline.png',
    '../visualizations/06_correlation_heatmaps.png'
]
for vf in vis_files:
    if os.path.exists(vf):
        print(f"File: {os.path.basename(vf)}")
        display(Image(filename=vf))
